# CVE Scanner — Multi-turn Security Vulnerability Search

## Overview

This notebook builds a security agent that scans a dependency list for recent CVEs using live web search. The agent issues one focused search per package, then synthesizes severity across all results into a prioritized remediation report.

**Why live search is critical here**: CVE databases update daily. A newly published exploit or patch changes the risk profile of a dependency within hours — training data is always stale for this.

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive (Jupyter Notebook) |
| AgentCore components | AgentCore Gateway |
| Agentic framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4 |
| Tutorial vertical | Security / DevOps |
| Example complexity | Intermediate |
| SDK used | boto3, strands-agents |

### Multi-turn Search Pattern

```
Input: requirements.txt
         │
         ▼
Parse packages + versions
         │
         ▼
For each package:
  Search 1: "CVE <package> <version> vulnerability"
  Search 2: "<package> latest version security patch"
         │
         ▼
Aggregate + score by CVSS severity
         │
         ▼
Output: Prioritized remediation report
  CRITICAL  requests 2.28.0 → CVE-XXXX, upgrade to 2.32.3
  HIGH      urllib3 1.26.5  → CVE-YYYY, upgrade to 2.2.2
```

## Prerequisites

Complete `01-gateway-setup-and-raw-mcp` first and export the environment variables it prints.

### Required IAM permissions

~~~json
{
  "Effect": "Allow",
  "Action": "bedrock:InvokeModel",
  "Resource": "*"
}
~~~

## 1. Install Dependencies

In [ ]:
!pip install --upgrade -r requirements.txt --quiet

## 2. Configuration and Setup

In [ ]:
import os
import requests
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
GATEWAY_URL = os.environ["AGENTCORE_GATEWAY_URL"]
COGNITO_DOMAIN = os.environ["COGNITO_DOMAIN"]
CLIENT_ID = os.environ["COGNITO_CLIENT_ID"]
CLIENT_SECRET = os.environ["COGNITO_CLIENT_SECRET"]
SCOPE_STRING = os.environ["COGNITO_SCOPE"]
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"


def get_token():
    url = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
    resp = requests.post(
        url,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
              "client_secret": CLIENT_SECRET, "scope": SCOPE_STRING}
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def create_transport():
    return streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {get_token()}"})


print("Configuration loaded ✓")

## 3. Define the Dependency List

This is the list of packages to scan. Update it with your actual dependencies.

In [ ]:
# Sample dependency list — replace with your actual requirements
DEPENDENCIES = [
    "requests==2.28.0",
    "urllib3==1.26.5",
    "cryptography==3.4.8",
    "pillow==9.0.0",
    "django==3.2.0"
]

print(f"Packages to scan ({len(DEPENDENCIES)}):")
for dep in DEPENDENCIES:
    print(f"  {dep}")

## 4. CVE Scanner Agent

The agent uses the CVE Scanner system prompt, which instructs it to search for each dependency separately and return a structured severity report.

In [ ]:
CVE_SYSTEM_PROMPT = """You are a security analyst agent with access to real-time web search.

When given a list of Python dependencies, you will:
1. For each package and version, search for known CVEs and security advisories
2. Search for the latest patched version of each package
3. Assign a severity level: CRITICAL, HIGH, MEDIUM, LOW, or NONE
4. Produce a remediation report sorted by severity

SEARCH STRATEGY:
- Use queries like "CVE <package> <version>" and "<package> security vulnerability 2025 2026"
- Keep each query under 200 characters
- Search each package independently before synthesizing

OUTPUT FORMAT:
For each package, output a row:
  <SEVERITY> | <package>==<version> | <CVE-IDs if found> | Upgrade to: <latest-safe-version> | <brief description>

End with a summary of total findings by severity level.
"""

mcp_client = MCPClient(create_transport)
print("CVE scanner agent ready ✓")

## 5. Run the CVE Scan

The agent will issue multiple searches — one per package — and synthesize a prioritized report.

In [ ]:
dep_list = "\n".join(f"  - {d}" for d in DEPENDENCIES)
scan_prompt = f"""Scan the following Python dependencies for security vulnerabilities and CVEs.
Search for each package individually, then produce a prioritized remediation report.

Dependencies:
{dep_list}
"""

print("Starting CVE scan...")
print("-" * 60)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.3, max_tokens=2048)
    agent = Agent(model=model, tools=tools, system_prompt=CVE_SYSTEM_PROMPT)
    response = agent(scan_prompt)

print("\n" + "=" * 60)
print("CVE SCAN REPORT")
print("=" * 60)
if hasattr(response, "message"):
    for block in response.message.get("content", []):
        if block.get("text"):
            print(block["text"])
else:
    print(str(response))

## 6. Scan a Custom requirements.txt

Pass the content of your actual `requirements.txt` file to scan your real dependencies.

In [ ]:
# To scan your own requirements.txt:
# with open("path/to/your/requirements.txt") as f:
#     requirements_content = f.read()

# custom_prompt = f"""Scan these Python dependencies for CVEs:\n\n{requirements_content}"""
# with mcp_client:
#     tools = mcp_client.list_tools_sync()
#     model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.3, max_tokens=2048)
#     agent = Agent(model=model, tools=tools, system_prompt=CVE_SYSTEM_PROMPT)
#     response = agent(custom_prompt)

print("Uncomment the cells above to scan your own requirements.txt")

## Conclusion

In this notebook you built a security scanning agent that:
- Issues one targeted search per dependency (multi-turn pattern)
- Finds CVEs that postdate any model training cutoff
- Produces a prioritized remediation report

The key insight: **each search is informed by the dependency from the parsed list** — this is the "search for each item" multi-turn pattern.

**Next**: See `02-earnings-brief.ipynb` for the chained search pattern where each result informs the next query.